# 07 — LOB Recorder (V2: BTC/USD Live Orderbook)

Records real-time Level 2 orderbook snapshots from Alpaca's WebSocket data stream
for BTC/USD. Snapshots are saved as compressed Parquet files for use in v3 training
and live inference via `strategy._get_live_lob_features()`.

**V2 CHANGE:** Uses `CryptoDataStream` (Alpaca WebSocket) instead of polling.
Captures top 3 bid/ask levels per bar.

**Output schema (per row):**
```
timestamp | bid_price_1 | bid_size_1 | bid_price_2 | ... | ask_size_3
```

**Output:** `/content/drive/MyDrive/algo_trader/data/lob/BTC_USD_lob_{date}.parquet`

**Duration:** Set `RECORD_MINUTES` below (default 1440 = 24 hours).

> Runs continuously until interrupted or `RECORD_MINUTES` elapsed.  
> Re-run daily to accumulate LOB training data for v3 real-LOB mode.

In [ ]:
!pip install -q alpaca-py pyarrow pandas tqdm

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
LOB_DIR = '/content/drive/MyDrive/algo_trader/data/lob'
os.makedirs(LOB_DIR, exist_ok=True)
print(f'LOB output directory: {LOB_DIR}')

In [ ]:
# Load API credentials from Colab Secrets (never hard-code)
ALPACA_API_KEY    = userdata.get('ALPACA_API_KEY')
ALPACA_SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise RuntimeError('Add ALPACA_API_KEY and ALPACA_SECRET_KEY to Colab Secrets first.')
print('Credentials loaded ✓')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
PAIR              = 'BTC/USD'          # Alpaca data-API format
RECORD_MINUTES    = 1440               # Recording duration (1440 = 24 hours)
SNAPSHOT_INTERVAL = 60                 # Seconds between LOB snapshots (1 per minute)
TOP_LEVELS        = 3                  # Number of bid/ask levels to capture

from datetime import datetime, timezone
DATE_STR = datetime.now(timezone.utc).strftime('%Y%m%d')
SAFE_NAME = PAIR.replace('/', '_')
OUT_PATH  = f'{LOB_DIR}/{SAFE_NAME}_lob_{DATE_STR}.parquet'

print(f'Pair:              {PAIR}')
print(f'Duration:          {RECORD_MINUTES} minutes')
print(f'Snapshot interval: {SNAPSHOT_INTERVAL} s')
print(f'Top LOB levels:    {TOP_LEVELS}')
print(f'Output file:       {OUT_PATH}')

In [ ]:
# ── LOB Recording via Alpaca REST polling (Colab-compatible) ──────────────────
# Note: Alpaca CryptoDataStream WebSocket requires a persistent async loop,
# which conflicts with Colab's IPython event loop.  We use the REST orderbook
# endpoint (CryptoLatestOrderbookRequest) polled every SNAPSHOT_INTERVAL seconds
# instead — this is identical data to the WebSocket snapshot feed.

import time
import asyncio
import pandas as pd
from datetime import datetime, timezone, timedelta
from tqdm.notebook import tqdm

from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoLatestOrderbookRequest

client = CryptoHistoricalDataClient(
    api_key    = ALPACA_API_KEY,
    secret_key = ALPACA_SECRET_KEY,
)

snapshots = []       # list of dicts, one per poll
end_time  = datetime.now(timezone.utc) + timedelta(minutes=RECORD_MINUTES)
n_expected = RECORD_MINUTES * 60 // SNAPSHOT_INTERVAL

print(f'Recording LOB for {RECORD_MINUTES} min '
      f'(~{n_expected} snapshots).  Interrupt kernel to stop early.')

pbar = tqdm(total=n_expected, desc='LOB snapshots')

try:
    while datetime.now(timezone.utc) < end_time:
        t_start = time.monotonic()
        ts_utc  = datetime.now(timezone.utc)

        try:
            req  = CryptoLatestOrderbookRequest(symbol_or_symbols=PAIR)
            resp = client.get_crypto_latest_orderbook(req)
            book = resp[PAIR]

            row = {'timestamp': ts_utc}
            for i, (bid, ask) in enumerate(
                zip(book.bids[:TOP_LEVELS], book.asks[:TOP_LEVELS]), start=1
            ):
                row[f'bid_price_{i}'] = float(bid.p)
                row[f'bid_size_{i}']  = float(bid.s)
                row[f'ask_price_{i}'] = float(ask.p)
                row[f'ask_size_{i}']  = float(ask.s)

            # Pad missing levels with NaN if book is shallow
            for i in range(len(book.bids[:TOP_LEVELS]) + 1, TOP_LEVELS + 1):
                for side in ['bid', 'ask']:
                    for field in ['price', 'size']:
                        row[f'{side}_{field}_{i}'] = float('nan')

            snapshots.append(row)
            pbar.update(1)

        except Exception as e:
            print(f'\nPoll error at {ts_utc.strftime("%H:%M:%S UTC")}: {e}')

        # Sleep for remainder of interval
        elapsed = time.monotonic() - t_start
        sleep_for = max(0.0, SNAPSHOT_INTERVAL - elapsed)
        time.sleep(sleep_for)

except KeyboardInterrupt:
    print('\nRecording interrupted by user.')
finally:
    pbar.close()

print(f'\nCaptured {len(snapshots):,} snapshots.')

In [ ]:
# ── Save LOB snapshots to Parquet ─────────────────────────────────────────────
if not snapshots:
    print('⚠️  No snapshots captured — nothing to save.')
else:
    lob_df = pd.DataFrame(snapshots)
    lob_df  = lob_df.set_index('timestamp')
    lob_df.index = pd.to_datetime(lob_df.index, utc=True)

    # Validate
    assert (lob_df['bid_price_1'] > 0).all(), 'Invalid bid prices'
    assert (lob_df['ask_price_1'] > 0).all(), 'Invalid ask prices'
    assert (lob_df['ask_price_1'] > lob_df['bid_price_1']).all(), 'Crossed book detected'

    lob_df.to_parquet(OUT_PATH, compression='snappy')

    print('=== LOB SAVE SUMMARY ===')
    print(f'  Rows saved:    {len(lob_df):,}')
    print(f'  Start:         {lob_df.index.min()}')
    print(f'  End:           {lob_df.index.max()}')
    print(f'  Avg spread:    ${(lob_df["ask_price_1"] - lob_df["bid_price_1"]).mean():.2f}')
    print(f'  Output file:   {OUT_PATH}')
    print(f'\n✅ LOB data saved. Use in 02_feature_engineering.ipynb for v3 real-LOB mode.')

In [ ]:
# ── LOB Data Quality Report ────────────────────────────────────────────────────
if 'lob_df' in dir() and len(lob_df) > 0:
    spread = lob_df['ask_price_1'] - lob_df['bid_price_1']
    spread_pct = spread / lob_df['bid_price_1'] * 100

    print('=== LOB QUALITY REPORT ===')
    print(f'  Total snapshots:    {len(lob_df):,}')
    print(f'  Spread (USD):       mean=${spread.mean():.2f}  max=${spread.max():.2f}')
    print(f'  Spread (bps):       mean={spread_pct.mean()*100:.1f}  max={spread_pct.max()*100:.1f}')
    print(f'  Bid depth (L1):     mean={lob_df["bid_size_1"].mean():.4f} BTC')
    print(f'  Ask depth (L1):     mean={lob_df["ask_size_1"].mean():.4f} BTC')

    nan_rows = lob_df.isna().any(axis=1).sum()
    print(f'  Rows with NaN:      {nan_rows} (shallow book at those moments)')

    # Flag if spread > 25 bps on average (expected range: 1–5 bps for BTC)
    if spread_pct.mean() > 0.05:
        print('  ⚠️  Average spread > 5 bps — check if data was recorded during thin market hours.')
    else:
        print('  ✅ Spread within normal BTC range.')
else:
    print('Run recording cell first.')